In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
# =========================
# 1. Paths
# =========================

LIAR_PATH = "/content/drive/MyDrive/Senior_Project/00_Data/cleaned/Standardized/LIAR/liar_standard.csv"
COVID_PATH = "/content/drive/MyDrive/Senior_Project/00_Data/cleaned/Standardized/COVID/covid_standard.csv"
SEN_PATH = "/content/drive/MyDrive/Senior_Project/00_Data/cleaned/Sensational/sensational_dataset.csv"

OUT_DIR = "/content/drive/MyDrive/Senior_Project/00_Data/cleaned/Final_Short_Form"
LIAR_DIR = f"{OUT_DIR}/LIAR"
COVID_DIR = f"{OUT_DIR}/COVID"
SEN_DIR = f"{OUT_DIR}/Sensational"
COMBINED_DIR = f"{OUT_DIR}/Combined"
LIAR_COVID_DIR = f"{OUT_DIR}/LIAR+COVID"

for path in [LIAR_DIR, COVID_DIR, SEN_DIR, COMBINED_DIR, LIAR_COVID_DIR]:
    os.makedirs(path, exist_ok=True)

In [ ]:
# =========================
# 2. Load datasets
# =========================

liar_df = pd.read_csv(LIAR_PATH)
covid_df = pd.read_csv(COVID_PATH)
sen_df = pd.read_csv(SEN_PATH)

liar_df["source"] = "liar"
covid_df["source"] = "covid"
sen_df["source"] = "sensational"

In [ ]:
# =========================
# 3. Standardize columns
# =========================

vader_cols = ["vader_neg", "vader_neu", "vader_pos", "vader_compound"]

standard_cols = [
    "ID", "text", "is_fake", "source",
    "vader_neg", "vader_neu", "vader_pos", "vader_compound"
]

liar_df = liar_df[standard_cols].copy()
covid_df = covid_df[standard_cols].copy()
sen_df = sen_df[standard_cols].copy()

for df in [liar_df, covid_df, sen_df]:
    df["text"] = df["text"].astype(str).str.strip()
    df["is_fake"] = df["is_fake"].astype(int)
    df.dropna(subset=["text", "is_fake"], inplace=True)
    df = df[df["text"] != ""]

In [ ]:
# =========================
# 3.5 Add rhetorical / linguistic features
# =========================

import re

def add_rhetorical_features(df):
    df = df.copy()
    text = df["text"].astype(str)

    # Basic length features
    df["word_count"] = text.apply(lambda x: len(x.split()))
    df["char_count"] = text.apply(len)

    # Punctuation / emphasis
    df["exclamation_count"] = text.apply(lambda x: x.count("!"))
    df["question_count"] = text.apply(lambda x: x.count("?"))
    df["all_caps_count"] = text.apply(
        lambda x: sum(
            1 for word in x.split()
            if word.isupper()
            and len(word) > 1
            and word != "[ENTITY]"
        )
    )

    # Rhetorical word groups
    certainty_words = [
        "always", "never", "definitely", "proven", "guaranteed",
        "undeniable", "everyone", "nobody", "all", "none"
    ]

    urgency_words = [
        "breaking", "urgent", "now", "immediately", "alert",
        "warning", "before", "latest"
    ]

    attack_words = [
        "fake", "lie", "lies", "liar", "corrupt", "scam",
        "fraud", "hoax", "attack", "exposed"
    ]

    second_person_words = ["you", "your", "yours"]

    def count_words(x, word_list):
        x = x.lower()
        return sum(len(re.findall(rf"\b{re.escape(word)}\b", x)) for word in word_list)

    df["certainty_count"] = text.apply(lambda x: count_words(x, certainty_words))
    df["urgency_count"] = text.apply(lambda x: count_words(x, urgency_words))
    df["attack_count"] = text.apply(lambda x: count_words(x, attack_words))
    df["second_person_count"] = text.apply(lambda x: count_words(x, second_person_words))

    return df

liar_df = add_rhetorical_features(liar_df)
covid_df = add_rhetorical_features(covid_df)
sen_df = add_rhetorical_features(sen_df)

In [ ]:
# =========================
# 3.6 Save feature-complete full datasets
# These include text, labels, VADER, and rhetorical features
# =========================

liar_df.to_csv(f"{LIAR_DIR}/liar_features_full.csv", index=False)
covid_df.to_csv(f"{COVID_DIR}/covid_features_full.csv", index=False)
sen_df.to_csv(f"{SEN_DIR}/sensational_features_full.csv", index=False)

print("Saved feature-complete full datasets.")

Saved feature-complete full datasets.


In [ ]:
# =========================
# 4. Create full combined dataset + combined_ID
# =========================

combined_full = pd.concat(
    [liar_df, covid_df, sen_df],
    ignore_index=True
).reset_index(drop=True)

combined_full["combined_ID"] = range(1, len(combined_full) + 1)

# mapping lets individual splits inherit the same combined_ID
mapping = combined_full[["ID", "source", "combined_ID"]].copy()

rhetoric_cols = [
    "word_count",
    "char_count",
    "exclamation_count",
    "question_count",
    "all_caps_count",
    "certainty_count",
    "urgency_count",
    "attack_count",
    "second_person_count"
]

combined_cols = [
    "combined_ID", "text", "is_fake", "source",
    "vader_neg", "vader_neu", "vader_pos", "vader_compound"
] + rhetoric_cols

combined_full_final = combined_full[combined_cols].copy()

In [ ]:
# =========================
# 5. Split each individual dataset
# =========================

train_liar, test_liar = train_test_split(
    liar_df,
    test_size=0.2,
    random_state=42,
    stratify=liar_df["is_fake"]
)

train_covid, test_covid = train_test_split(
    covid_df,
    test_size=0.2,
    random_state=42,
    stratify=covid_df["is_fake"]
)

train_sen, test_sen = train_test_split(
    sen_df,
    test_size=0.2,
    random_state=42,
    stratify=sen_df["is_fake"]
)

In [ ]:
# =========================
# 5.5 Create LIAR + COVID combined dataset + splits + liar_covid_ID
# =========================

liar_covid_full = pd.concat(
    [liar_df, covid_df],
    ignore_index=True
).reset_index(drop=True)

# Create unique ID for LIAR+COVID combined dataset
liar_covid_full["liar_covid_ID"] = range(1, len(liar_covid_full) + 1)

liar_covid_cols = [
    "liar_covid_ID", "text", "is_fake", "source",
    "vader_neg", "vader_neu", "vader_pos", "vader_compound"
] + rhetoric_cols

liar_covid_full_final = liar_covid_full[liar_covid_cols].copy()

print("\nLIAR + COVID full shape:", liar_covid_full_final.shape)

# Split AFTER liar_covid_ID is created, so the split keeps the same IDs
train_liar_covid, test_liar_covid = train_test_split(
    liar_covid_full,
    test_size=0.2,
    random_state=42,
    stratify=liar_covid_full["is_fake"]
)

train_liar_covid_final = train_liar_covid[liar_covid_cols].copy().reset_index(drop=True)
test_liar_covid_final = test_liar_covid[liar_covid_cols].copy().reset_index(drop=True)

print("Train LIAR+COVID:", train_liar_covid_final.shape)
print("Test LIAR+COVID:", test_liar_covid_final.shape)

# Sanity check
overlap_lc = set(train_liar_covid_final["liar_covid_ID"]).intersection(
    set(test_liar_covid_final["liar_covid_ID"])
)

print("Overlapping LIAR+COVID IDs:", len(overlap_lc))

assert len(overlap_lc) == 0, "Data leakage: same liar_covid_ID appears in train and test"
assert train_liar_covid_final["liar_covid_ID"].is_unique, "Duplicate liar_covid_IDs in train"
assert test_liar_covid_final["liar_covid_ID"].is_unique, "Duplicate liar_covid_IDs in test"


LIAR + COVID full shape: (20395, 17)
Train LIAR+COVID: (16316, 17)
Test LIAR+COVID: (4079, 17)
Overlapping LIAR+COVID IDs: 0


In [ ]:
# =========================
# 6. Build combined train/test from individual splits
# =========================

train_combined = pd.concat(
    [train_liar, train_covid, train_sen],
    ignore_index=True
)

test_combined = pd.concat(
    [test_liar, test_covid, test_sen],
    ignore_index=True
)

# map combined_ID into train/test
train_combined = train_combined.merge(
    mapping,
    on=["ID", "source"],
    how="left"
)

test_combined = test_combined.merge(
    mapping,
    on=["ID", "source"],
    how="left"
)

# make sure mapping worked
assert train_combined["combined_ID"].notna().all(), "Missing combined_ID in train_combined"
assert test_combined["combined_ID"].notna().all(), "Missing combined_ID in test_combined"

train_combined_final = train_combined[combined_cols].copy()
test_combined_final = test_combined[combined_cols].copy()

In [ ]:
# =========================
# 7. Save individual splits
# =========================

train_liar.to_csv(f"{LIAR_DIR}/train_liar.csv", index=False)
test_liar.to_csv(f"{LIAR_DIR}/test_liar.csv", index=False)

train_covid.to_csv(f"{COVID_DIR}/train_covid.csv", index=False)
test_covid.to_csv(f"{COVID_DIR}/test_covid.csv", index=False)

train_sen.to_csv(f"{SEN_DIR}/train_sensational.csv", index=False)
test_sen.to_csv(f"{SEN_DIR}/test_sensational.csv", index=False)

# LIAR + COVID combined
liar_covid_full_final.to_csv(f"{LIAR_COVID_DIR}/liar_covid_features_full.csv", index=False)
train_liar_covid_final.to_csv(f"{LIAR_COVID_DIR}/train_liar_covid.csv", index=False)
test_liar_covid_final.to_csv(f"{LIAR_COVID_DIR}/test_liar_covid.csv", index=False)

In [ ]:
# =========================
# 8. Save combined files
# =========================

combined_full_final.to_csv(f"{COMBINED_DIR}/combined_features_full.csv", index=False)
train_combined_final.to_csv(f"{COMBINED_DIR}/train_combined.csv", index=False)
test_combined_final.to_csv(f"{COMBINED_DIR}/test_combined.csv", index=False)

In [ ]:
# =========================
# 9. Sanity checks
# =========================

print("Full combined:", combined_full_final.shape)
print("Train combined:", train_combined_final.shape)
print("Test combined:", test_combined_final.shape)

print("\nTrain + test equals full?")
print(len(train_combined_final) + len(test_combined_final) == len(combined_full_final))

overlap = set(train_combined_final["combined_ID"]).intersection(
    set(test_combined_final["combined_ID"])
)

print("\nOverlapping combined_IDs:", len(overlap))
print("Duplicate train IDs:", train_combined_final["combined_ID"].duplicated().sum())
print("Duplicate test IDs:", test_combined_final["combined_ID"].duplicated().sum())

print("\nCombined full source counts:")
print(combined_full_final["source"].value_counts())

print("\nTrain source counts:")
print(train_combined_final["source"].value_counts())

print("\nTest source counts:")
print(test_combined_final["source"].value_counts())

print("\nTrain labels:")
print(train_combined_final["is_fake"].value_counts())

print("\nTest labels:")
print(test_combined_final["is_fake"].value_counts())

assert len(overlap) == 0, "Data leakage: same combined_ID appears in train and test"
assert train_combined_final["combined_ID"].is_unique, "Duplicate combined_IDs in train"
assert test_combined_final["combined_ID"].is_unique, "Duplicate combined_IDs in test"

print("\nAll checks passed.")

Full combined: (21567, 17)
Train combined: (17252, 17)
Test combined: (4315, 17)

Train + test equals full?
True

Overlapping combined_IDs: 0
Duplicate train IDs: 0
Duplicate test IDs: 0

Combined full source counts:
source
liar           10796
covid           9599
sensational     1172
Name: count, dtype: int64

Train source counts:
source
liar           8636
covid          7679
sensational     937
Name: count, dtype: int64

Test source counts:
source
liar           2160
covid          1920
sensational     235
Name: count, dtype: int64

Train labels:
is_fake
0    9716
1    7536
Name: count, dtype: int64

Test labels:
is_fake
0    2431
1    1884
Name: count, dtype: int64

All checks passed.


In [ ]:
print("\nLIAR + COVID label distribution:")
print(liar_covid_full["is_fake"].value_counts())

print("\nTrain LIAR+COVID labels:")
print(train_liar_covid["is_fake"].value_counts())

print("\nTest LIAR+COVID labels:")
print(test_liar_covid["is_fake"].value_counts())


LIAR + COVID label distribution:
is_fake
0    11561
1     8834
Name: count, dtype: int64

Train LIAR+COVID labels:
is_fake
0    9249
1    7067
Name: count, dtype: int64

Test LIAR+COVID labels:
is_fake
0    2312
1    1767
Name: count, dtype: int64
